# 🧹 Module 14 — Data Cleaning robuste
> **Bootcamp Data Analyst — From Zero to Hero** | Niveau Intermédiaire · Module 14

---

## 🎯 Ce que tu seras capable de faire à la fin de ce module

- Nettoyer des données textuelles complexes avec des **expressions régulières** (regex)
- Lire une trace d'erreur Python (traceback) et savoir où chercher le problème
- Isoler un bug avec une méthode simple et reproductible
- Documenter un script proprement : docstrings, commentaires utiles, README minimal

---

> ⏱️ **Durée estimée** : 90 minutes
> 🛠️ **Outils** : Python (module `re`)
> 📌 **Prérequis** : Module 13 terminé · Modules 10-12 (Python) du Niveau Débutant

---
## 1. Pourquoi aller au-delà de `SUPPRESPACE`/`.str.strip()`

Au Niveau Débutant, tu as déjà nettoyé des données : espaces parasites (`SUPPRESPACE`, `.str.strip()`), valeurs manquantes. Mais les vraies données administratives et commerciales ivoiriennes posent des problèmes plus tordus :

| Donnée brute | Le problème |
|---|---|
| `"07 12 34 56 78"`, `"0712345678"`, `"+225 0712345678"`, `"225-07-12-34-56-78"` | Un même numéro de téléphone, 4 formats différents |
| `"1 250 000"`, `"1.250.000"`, `"1250000 FCFA"` | Un même montant, séparateurs de milliers différents |
| `"ABIDJAN"`, `"Abidjan "`, `"abidjan"`, `" Abidjan"` | Une même ville, casse et espaces incohérents |
| `"Kouassi, Jean (07-12-2023)"` | Plusieurs informations mélangées dans une seule cellule texte |

`.str.strip()` ne résout que le dernier problème dans la 3ᵉ ligne. Pour tout le reste, il faut décrire un **motif** (pattern) — c'est exactement ce que fait une expression régulière.

---
## 2. Les expressions régulières (regex)

Une regex est un texte qui décrit un **motif** à rechercher. En Python, le module `re` fournit les fonctions de base.

### Les briques essentielles

| Motif | Signifie | Exemple |
|---|---|---|
| `\d` | Un chiffre | `\d\d\d` matche `"123"` |
| `\s` | Un espace (space, tab...) | |
| `+` | 1 fois ou plus | `\d+` matche `"7"`, `"12"`, `"345678"` |
| `*` | 0 fois ou plus | |
| `?` | 0 ou 1 fois | |
| `[...]` | Un caractère parmi une liste | `[A-Za-z]` = une lettre |
| `^` / `$` | Début / fin de chaîne | |
| `(...)` | Un groupe (pour extraire un morceau précis) | |

### `re.search`, `re.sub`, et les groupes

```python
import re

# 1. Vérifier qu'un texte contient un motif
texte = "Contact : 07 12 34 56 78"
if re.search(r"\d{2}(\s?\d{2}){4}", texte):
    print("Numéro trouvé")

# 2. Nettoyer : remplacer tout ce qui n'est pas un chiffre par rien
numero_brut = "+225 07 12 34 56 78"
numero_propre = re.sub(r"\D", "", numero_brut)
print(numero_propre)
# "2250712345678"

# 3. Extraire une information précise avec un groupe
montant = "1 250 000 FCFA"
chiffres = re.sub(r"[^\d]", "", montant)
print(int(chiffres))
# 1250000
```

### Cas concret : uniformiser des numéros de téléphone ivoiriens

```python
import re
import pandas as pd

numeros = ["07 12 34 56 78", "0712345678", "+225 0712345678", "225-07-12-34-56-78"]

def uniformiser(numero):
    chiffres = re.sub(r"\D", "", numero)       # ne garder que les chiffres
    chiffres = chiffres[-10:]                    # garder les 10 derniers (retire le 225 éventuel)
    return chiffres

propres = [uniformiser(n) for n in numeros]
print(propres)
# ['0712345678', '0712345678', '0712345678', '0712345678']
```

Les 4 formats deviennent identiques — indispensable avant de dédupliquer une base de contacts ou de croiser deux fichiers sur le numéro de téléphone.

<details>
<summary>👉 À toi de jouer</summary>

Écris une regex qui extrait uniquement le montant numérique de `"Salaire net : 450.000 FCFA/mois"`. Indice : `re.sub(r"[^\d]", "", texte)` retire tout sauf les chiffres — mais attention, ça retire aussi les points utilisés comme séparateurs de milliers, ce qui est justement ce que tu veux ici.
</details>

---
## 3. Déboguer un script qui plante

### Lire une trace d'erreur (traceback)

Quand un script Python plante, il affiche une **traceback** — ne la fuis pas, elle te dit exactement où chercher.

```python
import pandas as pd

df = pd.DataFrame({"salaire": ["450000", "380 000", "non renseigné"]})
df["salaire_num"] = df["salaire"].astype(int)
```

```
Traceback (most recent call last):
  File "script.py", line 3, in <module>
    df["salaire_num"] = df["salaire"].astype(int)
ValueError: invalid literal for int() with base 10: '380 000'
```

Trois informations essentielles, à lire **de bas en haut** :
1. **Le type d'erreur** (`ValueError`) et son message : `int()` ne sait pas convertir `'380 000'` (l'espace bloque la conversion)
2. **La ligne exacte** qui a planté (`line 3`)
3. **La pile d'appels** au-dessus, si l'erreur vient d'une fonction imbriquée

### Méthode pour isoler un bug

1. **Reproduis en petit** : teste la ligne fautive sur une seule valeur (`int("380 000")`) plutôt que sur tout un DataFrame de 10 000 lignes.
2. **Affiche l'état intermédiaire** : un `print(df["salaire"].unique())` avant la conversion révèle souvent le format inattendu (ici, l'espace).
3. **Corrige à la source** : nettoie avec une regex avant de convertir, plutôt que de bricoler un `try/except` qui cache le problème.

```python
df["salaire_num"] = df["salaire"].apply(
    lambda s: int(re.sub(r"\D", "", s)) if re.search(r"\d", s) else None
)
print(df)
#           salaire  salaire_num
# 0          450000     450000.0
# 1         380 000     380000.0
# 2  non renseigné          NaN
```

> 💡 Un `try/except` qui avale l'erreur sans rien dire est le pire réflexe de debugging — tu perds l'information qui t'aurait dit où est le problème. Log ou affiche toujours ce qui a été ignoré.

---
## 4. Documenter son travail

Un script qui fonctionne aujourd'hui et qu'on ne comprend plus dans 3 mois n'est pas un script fini.

### Docstring — expliquer une fonction

```python
def uniformiser_numero(numero: str) -> str:
    """Uniformise un numéro de téléphone ivoirien en 10 chiffres sans préfixe.

    Gère les formats : espaces, tirets, +225, préfixe 225.
    Exemple : "+225 07 12 34 56 78" -> "0712345678"
    """
    chiffres = re.sub(r"\D", "", numero)
    return chiffres[-10:]
```

### Commentaires utiles vs bruit

```python
# ❌ Commentaire inutile — répète ce que le code dit déjà
total = df["montant"].sum()  # calcule la somme de montant

# ✅ Commentaire utile — explique un choix non-évident
total = df["montant"].sum()  # les valeurs négatives = remboursements, on les garde volontairement
```

> Un bon commentaire répond à "pourquoi ?", jamais à "quoi ?" — le code répond déjà à "quoi".

### README minimal pour un projet d'analyse

```markdown
# Analyse RH — Bamba & Associés

## Contenu
- `nettoyage.py` — uniformise les numéros de téléphone et montants
- `analyse.py` — calculs et graphiques

## Comment l'exécuter
1. `pip install -r requirements.txt`
2. `python nettoyage.py` puis `python analyse.py`

## Données
Source : export RH interne (confidentiel, non inclus dans ce dépôt)
```

C'est le même réflexe que le README GitHub vu au Module 13 — c'est souvent la première chose qu'un∙e collègue ou un∙e recruteur∙se lit.

---
## ✅ Résumé du module

| Concept | Ce qu'il faut retenir |
|---|---|
| **Regex** | Décrit un motif à rechercher/remplacer dans du texte |
| **`\d`, `\s`, `+`, `*`, `[...]`** | Briques de base pour construire un motif |
| **`re.search`** | Vérifier qu'un motif existe dans un texte |
| **`re.sub`** | Remplacer ce qui matche un motif (souvent : tout retirer sauf les chiffres) |
| **Traceback** | Se lit de bas en haut : type d'erreur, message, ligne fautive |
| **Isoler un bug** | Reproduire en petit, afficher l'état intermédiaire, corriger à la source |
| **`try/except` silencieux** | Le pire réflexe — il cache l'information dont tu as besoin |
| **Docstring** | Explique ce que fait une fonction et comment l'utiliser |
| **Bon commentaire** | Répond à "pourquoi", jamais à "quoi" |
| **README** | Contenu, comment exécuter, source des données |

---

## 🧠 Quiz — Vérifie ta compréhension

---

**Q1.** Quelle regex retire tout ce qui n'est pas un chiffre d'une chaîne `numero` ?
- a) `re.sub(r"\d", "", numero)`
- b) `re.sub(r"\D", "", numero)`
- c) `re.search(r"\d+", numero)`

<details>
<summary>👉 Voir la réponse</summary>

✅ **b)** — `\D` (D majuscule) signifie "tout sauf un chiffre". `re.sub(r"\D", "", numero)` remplace chaque caractère non-numérique par rien, ne laissant que les chiffres. La réponse a) ferait l'inverse (supprimerait les chiffres), et c) ne fait que chercher, pas remplacer.
</details>

---

**Q2.** Ton script plante avec `ValueError: invalid literal for int() with base 10: '380 000'`. Que fais-tu en premier ?
- a) J'entoure la ligne d'un `try/except: pass` pour que le script continue
- b) Je teste `int("380 000")` isolément pour comprendre exactement ce qui bloque
- c) Je relance le script en espérant que ça passe cette fois

<details>
<summary>👉 Voir la réponse</summary>

✅ **b)** — Reproduire le problème en isolant la valeur fautive te montre immédiatement que c'est l'espace qui bloque `int()`. Le `try/except: pass` (a) cache l'erreur sans la résoudre — tu auras juste une valeur manquante inexpliquée plus loin.
</details>

---

**Q3.** Lequel de ces commentaires est utile ?
- a) `# incrémente x de 1` au-dessus de `x += 1`
- b) `# on exclut les stagiaires : leur contrat n'a pas de date de fin, ça fausserait la moyenne` au-dessus d'un filtre
- c) `# fonction qui calcule quelque chose`

<details>
<summary>👉 Voir la réponse</summary>

✅ **b)** — Ce commentaire explique une décision non-évidente (*pourquoi* exclure les stagiaires) que le code seul ne révèle pas. Les options a) et c) répètent ou vagualisent ce que le code dit déjà.
</details>

---

## ➡️ Module suivant

Tu sais maintenant nettoyer des données réelles et déboguer proprement. Dans le **Module 15**, on approfondit pandas et on découvre DuckDB pour interroger des DataFrames en SQL.

> **→ Module 15 — Pandas approfondi + DuckDB**